# FraudLens — Data Cleaning

In [1]:
import os
import pandas as pd
import numpy as np

RAW_PATH = "../data/Base.csv"
OUTPUT_PATH = "../data/cleaned_base.csv"

df = pd.read_csv(RAW_PATH)
print(f"Loaded raw dataset shape: {df.shape}")
df.head()

Loaded raw dataset shape: (1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,...,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,...,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,...,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,...,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,...,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,...,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


## Handle Missing Values


In [2]:
num_cols_median = ['current_address_months_count', 'bank_months_count', 'session_length_in_minutes']
num_cols_zero = ['prev_address_months_count']
cat_cols_flag = ['device_distinct_emails_8w']

for col in num_cols_median:
    if (df[col] == -1).any():
        df[f'{col}_is_missing'] = (df[col] == -1).astype(int)
        valid_median = df.loc[df[col] != -1, col].median()
        df[col] = df[col].replace(-1, valid_median)
        print(f"{col}: imputed {df[f'{col}_is_missing'].sum()} missing values with median {valid_median}")

for col in num_cols_zero:
    if (df[col] == -1).any():
        df[f'{col}_is_missing'] = (df[col] == -1).astype(int)
        df[col] = df[col].replace(-1, 0)
        print(f"{col}: imputed {df[f'{col}_is_missing'].sum()} missing values with 0")

for col in cat_cols_flag:
    if (df[col] == -1).any():
        df[col] = df[col].astype(str).replace(['-1.0', '-1'], 'MISSING')
        print(f"{col}: recoded -1 values as 'MISSING' category")

current_address_months_count: imputed 4254 missing values with median 53.0
bank_months_count: imputed 253635 missing values with median 15.0
session_length_in_minutes: imputed 2015 missing values with median 5.122832390126673
prev_address_months_count: imputed 712920 missing values with 0
device_distinct_emails_8w: recoded -1 values as 'MISSING' category


##Check the `month` Column for Leakage

In [4]:
print(df['month'].describe())
print()
print("Fraud rate by month:")
print(df.groupby('month')['fraud_bool'].mean())

count    1000000.000000
mean           3.288674
std            2.209994
min            0.000000
25%            1.000000
50%            3.000000
75%            5.000000
max            7.000000
Name: month, dtype: float64

Fraud rate by month:
month
0    0.011326
1    0.009387
2    0.008746
3    0.009222
4    0.011371
5    0.011825
6    0.013405
7    0.014746
Name: fraud_bool, dtype: float64


## One-Hot Encode Categorical Columns

In [5]:
categorical_cols = ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os', 'device_distinct_emails_8w']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
print(f"Shape after encoding: {df.shape}")

Shape after encoding: (1000000, 54)


##  Remove Redundant Columns

In [6]:
if 'device_fraud_count' in df.columns:
    print(f"device_fraud_count unique values: {df['device_fraud_count'].unique()}")
    df = df.drop(columns=['device_fraud_count'])
    print("Dropped zero-variance column: 'device_fraud_count'")

device_fraud_count unique values: [0]
Dropped zero-variance column: 'device_fraud_count'


##  Final Validation

In [7]:
null_counts = df.isnull().sum()
assert null_counts.sum() == 0, f"Found nulls in: {null_counts[null_counts > 0]}"
print("No null values — passed.")

non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
assert len(non_numeric) == 0, f"Non-numeric columns remain: {non_numeric}"
print("All columns numeric — passed.")

print(f"\nFinal cleaned shape: {df.shape}")

No null values — passed.
All columns numeric — passed.

Final cleaned shape: (1000000, 53)


## Save Cleaned Dataset

In [8]:
output_dir = os.path.dirname(OUTPUT_PATH)
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)
    print(f"Created new directory: {output_dir}")

df.to_csv(OUTPUT_PATH, index=False)
print(f"Cleaned dataset saved to: {OUTPUT_PATH}")
print(f"Final saved shape: {df.shape}")

Cleaned dataset saved to: ../data/cleaned_base.csv
Final saved shape: (1000000, 53)
